# 01 - Data Acquisition & Preprocessing

This notebook fetches satellite (Sentinel-2), weather (ERA5), and soil (SoilGrids) data for the region of interest using Google Earth Engine. If GEE is not available, it generates synthetic data for offline development.

## 1. Setup and Authentication

In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
from pathlib import Path

# ROI: Kansas, USA (small agricultural area)
ROI_COORDS = [[-97.5, 38.5], [-97.0, 38.5], [-97.0, 39.0], [-97.5, 39.0]]
START_DATE = '2023-04-01'
END_DATE = '2023-07-31'
TARGET_YEAR = 2023

In [2]:
# Google Earth Engine authentication (run: earthengine authenticate)
try:
    import ee
    ee.Initialize()
    GEE_AVAILABLE = True
    print('Earth Engine initialized successfully.')
except Exception as e:
    GEE_AVAILABLE = False
    print(f'GEE not available: {e}. Using synthetic data.')

GEE not available: No module named 'ee'. Using synthetic data.


## 2. Sentinel-2 with Cloud Masking and 10-Day Composites

In [3]:
if GEE_AVAILABLE:
    # Load Sentinel-2 L2A, apply QA60 cloud mask, create composites
    roi = ee.Geometry.Polygon(ROI_COORDS + [ROI_COORDS[0]])
    collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(roi)
        .filterDate(START_DATE, END_DATE)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))
    
    def mask_clouds(img):
        qa = img.select('QA60')
        cloud_bit = 1 << 10
        cirrus_bit = 1 << 11
        mask = qa.bitwiseAnd(cloud_bit).eq(0).And(qa.bitwiseAnd(cirrus_bit).eq(0))
        return img.updateMask(mask)
    
    def add_indices(img):
        ndvi = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
        evi = img.expression(
            '2.5 * (NIR - RED) / (NIR + 6*RED - 7.5*BLUE + 1)',
            {'NIR': img.select('B8'), 'RED': img.select('B4'), 'BLUE': img.select('B2')}).rename('EVI')
        savi = img.expression(
            '1.5 * (NIR - RED) / (NIR + RED + 0.5)',
            {'NIR': img.select('B8'), 'RED': img.select('B4')}).rename('SAVI')
        return img.addBands([ndvi, evi, savi])
    
    masked = collection.map(mask_clouds).map(add_indices)
    composite = masked.median()
    print('Sentinel-2 composite created.')
else:
    print('Skipping GEE - using synthetic data.')

Skipping GEE - using synthetic data.


## 3. ERA5 Weather and SoilGrids

In [4]:
if GEE_AVAILABLE:
    era5 = ee.ImageCollection('ECMWF/ERA5/DAILY').filterBounds(roi).filterDate(START_DATE, END_DATE)
    print('ERA5 loaded.')
    # SoilGrids: use OPENLANDMAP or similar if available in GEE
    print('Soil data: use SoilGrids from ISRIC or GEE catalog.')

## 4. Extract Values per Field & Save

In [5]:
from src.synthetic_data import generate_synthetic_dataset
from src.utils import get_data_dir

# Use synthetic data (realistic relationships) when GEE extraction is complex
# In production: use ee.Image.sampleRegions() with field polygons
df = generate_synthetic_dataset(
    n_fields=100,
    start_date=START_DATE,
    end_date=END_DATE,
    target_year=TARGET_YEAR,
    seed=42
)

data_dir = get_data_dir()
data_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(data_dir / 'merged_data.csv', index=False)
print(f'Saved {len(df)} rows to {data_dir / "merged_data.csv"}')
df.head()

Saved 1300 rows to /Users/monurajj/Desktop/Satellite-Based Precision Agriculture/data/merged_data.csv


,field_id,date,B2,B3,B4,B8,B11,B12,NDVI,EVI,NDRE,SAVI,temperature,precipitation,solar_rad,soil_pH,soil_OC,soil_clay,yield
0,field_0,2023-04-01,0.084292,0.163191,0.085214,0.229573,0.228003,0.158474,0.215954,0.325524,0.587517,0.265761,19.070676,14.054420,117.442823,6.37454,1.047901,27.840633,8.0
1,field_0,2023-04-11,0.082234,0.153541,0.099689,0.200623,0.213554,0.153326,0.167704,0.213482,0.483869,0.189177,16.675105,14.134216,112.043217,6.37454,1.047901,27.840633,8.0
2,field_0,2023-04-21,0.072321,0.218811,0.001783,0.396434,0.231246,0.171154,0.494057,1.140980,0.993724,0.659058,18.673057,13.917463,116.641020,6.37454,1.047901,27.840633,8.0
3,field_0,2023-05-01,0.071957,0.205144,0.022285,0.355431,0.242236,0.182996,0.425718,0.877197,0.915914,0.569341,15.944022,12.838239,115.884329,6.37454,1.047901,27.840633,8.0
4,field_0,2023-05-11,0.082317,0.175107,0.067339,0.265322,0.232957,0.153855,0.275536,0.470500,0.698281,0.356656,16.749665,13.606272,116.930645,6.37454,1.047901,27.840633,8.0
